# 🎬 KINO Kaggle-motor (egen GPU) – reserv när Colab inte ansluter

Samma motor som Colab-versionen (samma server, samma endpoints och samma säkerhetsregler: minderårsspärr, åldersspärr 21 på ansikten, ansiktsbyte bara mellan era egna artister med samtycke, NSFW bara där KINO tillåter det). Servern hämtas från GitHub (`colab/kino_colab_server.py`), så rättningar gäller båda.

**Så gör du:**
1. kaggle.com → **Create → New Notebook** → **File → Import Notebook** → klistra in `https://github.com/pistpejl/kino/blob/main/kaggle/kino_kaggle_engine.ipynb`.
2. Högerpanelen **Settings**: **Accelerator → GPU T4 x2**, **Internet → On** (kräver telefonverifierat Kaggle-konto).
3. *Valfritt:* **Add-ons → Secrets** → lägg till `HF_TOKEN` (bara för FLUX 3 Image Klein, som är gated).
4. **Run All**. Sista cellerna visar **adress** och **nyckel** i en grön ruta.
5. I KINO: **Colab / Kaggle (egen GPU)** → klistra in adress + nyckel → **Spara & testa**.

Kaggle ger ca 30 GPU-timmar/vecka och max 12 h per session. Stäng sessionen (Stop) när du är klar.

## 1. Kolla GPU

In [ ]:
import os
os.makedirs('/tmp/kino', exist_ok=True); os.environ['HF_HOME'] = '/tmp/hf'  # /kaggle/working har bara 20 GB
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # T4 x2: motorn använder bara den första (cuda:0)
os.environ['KINO_PLATFORM'] = 'kaggle'
import torch, sys
print('Python', sys.version.split()[0], '| torch', torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit('INGEN GPU i den här sessionen. Högerpanelen Settings → Accelerator → välj "GPU T4 x2" '
                     '(sessionen startar om), sedan Run All igen. Kräver telefonverifierat konto och GPU-kvot kvar.')
print('GPU:', torch.cuda.get_device_name(0))
import urllib.request
try: urllib.request.urlopen('https://huggingface.co', timeout=10); print('Internet: på')
except Exception: raise SystemExit('Internet är av: Settings → Internet → On (kräver telefonverifiering).')

## 2. Installera (låsta versioner; Kaggles egen torch/CUDA används och installeras inte om)

In [ ]:
import sys
import os, subprocess
os.makedirs('/tmp/kino/logs', exist_ok=True)
def sh(cmd, log):
    """Kör ett skalkommando, all utdata till loggfil (inga röda rutor). Returnerar True/False i stället för att krascha."""
    with open(log, 'a') as f:
        f.write('$ ' + cmd + '\n'); f.flush()
        r = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode:
        print(f'MISSLYCKADES ({r.returncode}): {cmd[:90]}\n--- sista raderna i {log} ---')
        print(''.join(open(log, errors='replace').readlines()[-15:]))
    return r.returncode == 0
# Kaggles egen torch/numpy/fastapi/uvicorn/opencv/onnx används; bara det som saknas installeras. All utdata går till /tmp/kino/logs/install.log.
L = '/tmp/kino/logs/install.log'
ok = sh('pip install -q --progress-bar off diffusers==0.42.0 transformers==5.20.2 accelerate==1.15.0 "bitsandbytes>=0.50.2" spandrel==0.4.2 "sentencepiece>=0.2.1" ftfy '
        '"imageio>=2.37" "imageio-ffmpeg>=0.6" "fastapi>=0.115" "uvicorn>=0.34" "python-multipart>=0.0.20" "librosa>=0.11" "soundfile>=0.13.1" '
        'openai-whisper==20250625 onnxruntime-gpu "einops>=0.8.1" treetable==0.2.5 submitit==1.5.2 "omegaconf>=2.3" retrying==1.3.4', L)
# insightface byggs från källkod (behöver g++): egen rad så att ett byggfel inte stoppar resten
sh('pip install -q --progress-bar off insightface==0.7.3', L)
sh('pip install -q --progress-bar off --no-deps demucs==4.0.1 dora-search==0.1.12 julius==0.2.7 lameenc==1.8.1 openunmix==1.3.0', L)
sh('apt-get -qq install -y ffmpeg', L)
# Cloudflare quick tunnel (gratis, inget konto), låst version från Cloudflares officiella GitHub
sh('wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/download/2025.4.0/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared', L)
# Importtest i en ny process (samma som servern kommer att se)
mods = ['fastapi', 'uvicorn', 'multipart', 'diffusers', 'transformers', 'accelerate', 'insightface', 'onnxruntime', 'cv2', 'whisper', 'librosa', 'demucs.pretrained', 'spandrel']
r = subprocess.run([sys.executable if 'sys' in dir() else 'python', '-c', 'import importlib,sys\nfor m in sys.argv[1:]:\n  try: importlib.import_module(m)\n  except Exception as e: print(m, "->", type(e).__name__, str(e)[:150])'] + mods,
                   capture_output=True, text=True)
print(('VARNING, kan inte importeras:\n' + r.stdout) if r.stdout.strip() else 'Alla moduler importeras.')
print('cloudflared:', 'ok' if os.path.exists('/usr/local/bin/cloudflared') else 'SAKNAS')
print('Klart')

## 3. Hämta servern från GitHub (en källa för Colab och Kaggle)

In [ ]:
import urllib.request, os
os.makedirs('/tmp/kino', exist_ok=True)
try: urllib.request.urlretrieve('https://raw.githubusercontent.com/pistpejl/kino/main/colab/kino_colab_server.py', '/tmp/kino/kino_colab_server.py')
except Exception as e: raise SystemExit(f'Kunde inte hämta servern ({e}). Är Internet på? Settings → Internet → On.')
print('Server hämtad:', os.path.getsize('/tmp/kino/kino_colab_server.py'), 'byte')


## 4. (Valfritt) Ansiktsbyte
Hämtar inswapper_128.onnx och kontrollerar SHA-256. Misslyckas det: lägg filen som Kaggle-dataset (Add Data) och kopiera den till `/tmp/kino/models/`. Endast icke-kommersiellt bruk.

In [ ]:
import hashlib, os, urllib.request, glob, shutil
os.makedirs('/tmp/kino/models', exist_ok=True)
P = '/tmp/kino/models/inswapper_128.onnx'
SHA = 'e4a3f08c753cb72d04e10aa0f7dbe3deebbf39567d4ead6dce08e98aa49e16af'
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
try:
    if not os.path.exists(P):
        try: urllib.request.urlretrieve('https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx', P)
        except Exception as e: print('Nedladdning misslyckades:', e)
    if os.path.exists(P) and sha(P) != SHA:
        os.remove(P); print('Fel kontrollsumma: filen raderades.')
    if not os.path.exists(P):  # Kaggle har ingen uppladdningsruta: leta i tillagda dataset (Add Data)
        for f in glob.glob('/kaggle/input/**/inswapper_128.onnx', recursive=True):
            if sha(f) == SHA: shutil.copy(f, P); break
except Exception as e: print('Ansiktsbyte hoppas över:', e)
print('Ansiktsbyte redo:', os.path.exists(P), '' if os.path.exists(P) else '(valfritt – resten av motorn fungerar ändå)')

## 5. (Valfritt) Lipsync med LatentSync 1.5 (ca 10 min)

In [ ]:
import os, subprocess
os.makedirs('/tmp/kino/logs', exist_ok=True)
def sh(cmd, log):
    """Kör ett skalkommando, all utdata till loggfil (inga röda rutor). Returnerar True/False i stället för att krascha."""
    with open(log, 'a') as f:
        f.write('$ ' + cmd + '\n'); f.flush()
        r = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode:
        print(f'MISSLYCKADES ({r.returncode}): {cmd[:90]}\n--- sista raderna i {log} ---')
        print(''.join(open(log, errors='replace').readlines()[-15:]))
    return r.returncode == 0
L = '/tmp/kino/logs/latentsync.log'
def _ls():
    if not os.path.isdir('/tmp/kino/LatentSync/.git') and not sh('git clone -q https://github.com/bytedance/LatentSync /tmp/kino/LatentSync', L): return False
    if not sh('cd /tmp/kino/LatentSync && git checkout -q a229c3948406bc2cf6eaf4873e662e70c6a04746', L): return False
    sh('apt-get -qq install -y libgl1', L)
    if not sh('command -v uv || pip install -q uv', L): return False   # Kaggle har redan uv
    if not os.path.exists('/tmp/kino/ls_env/bin/python'):
        if not sh('uv venv -q /tmp/kino/ls_env --python 3.10 --seed', L): return False
    if not sh('uv pip install -q --python /tmp/kino/ls_env/bin/python -r /tmp/kino/LatentSync/requirements.txt', L): return False
    for f in ['latentsync_unet.pt', 'whisper/tiny.pt']:
        if not sh(f'python -c "from huggingface_hub import hf_hub_download as d; d(\'ByteDance/LatentSync-1.5\', \'{f}\', local_dir=\'/tmp/kino/LatentSync/checkpoints\')"', L): return False
    return True
try: ok = _ls()
except Exception as e: ok = False; print('Fel:', e)
if not ok and os.path.isdir('/tmp/kino/ls_env'):  # så att servern inte annonserar lipsync med en trasig miljö
    import shutil; shutil.rmtree('/tmp/kino/ls_env', ignore_errors=True)
print('LatentSync 1.5 klar.' if ok else 'LatentSync hoppas över (valfritt) – motorn startar ändå, bara lipsync saknas. Logg: ' + L)

## (Valfritt) Röstbyte med Seed-VC (ca 5 min)
Officiella repot `Plachtaa/seed-vc`, låst till commit `51383efd9210`, med sångmodellen (whisper-base, f0, 44,1 kHz) från Hugging Face `Plachta/Seed-VC`. Körs i en egen Python 3.10-miljö med egna låsta versioner (Seed-VC kräver transformers 4.46.3 och påverkar därför inte motorns diffusers 0.42.0 / transformers 5.20.2 / accelerate 1.15.0). **Bara röster ur din samtyckta röstlista** kan användas. Seed-VC är GPL-3.0.

In [ ]:
import os, subprocess
os.makedirs('/tmp/kino/logs', exist_ok=True)
def sh(cmd, log):
    """Kör ett skalkommando, all utdata till loggfil (inga röda rutor). Returnerar True/False i stället för att krascha."""
    with open(log, 'a') as f:
        f.write('$ ' + cmd + '\n'); f.flush()
        r = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode:
        print(f'MISSLYCKADES ({r.returncode}): {cmd[:90]}\n--- sista raderna i {log} ---')
        print(''.join(open(log, errors='replace').readlines()[-15:]))
    return r.returncode == 0
L = '/tmp/kino/logs/seedvc.log'
def _svc():
    if not os.path.isdir('/tmp/kino/seed-vc/.git') and not sh('git clone -q https://github.com/Plachtaa/seed-vc /tmp/kino/seed-vc', L): return False
    if not sh('cd /tmp/kino/seed-vc && git checkout -q 51383efd921027683c89e5348211d93ff12ac2a8', L): return False
    if not sh('command -v uv || pip install -q uv', L): return False   # Kaggle har redan uv
    if not os.path.exists('/tmp/kino/svc_env/bin/python'):
        if not sh('uv venv -q /tmp/kino/svc_env --python 3.10 --seed', L): return False
    if not sh('uv pip install -q --python /tmp/kino/svc_env/bin/python torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 numpy==1.26.4 scipy==1.13.1 librosa==0.10.2 soundfile==0.12.1 huggingface-hub==0.28.1 munch==4.0.0 einops==0.8.0 descript-audio-codec==1.0.0 pydub==0.25.1 transformers==4.46.3 accelerate==1.0.1 hydra-core==1.3.2 pyyaml==6.0.2 tqdm==4.67.1', L): return False
    # Hämtar vikterna en gång (kort testkonvertering med repots exempelfiler)
    return sh('cd /tmp/kino/seed-vc && /tmp/kino/svc_env/bin/python inference.py --source examples/source/source_s1.wav --target examples/reference/s1p1.wav '
              '--output /tmp/kino/svc_test --diffusion-steps 10 --f0-condition True --fp16 True', L)
try: ok = _svc()
except Exception as e: ok = False; print('Fel:', e)
if not ok and os.path.isdir('/tmp/kino/svc_env'):  # så att servern inte annonserar röstbyte med en trasig miljö
    import shutil; shutil.rmtree('/tmp/kino/svc_env', ignore_errors=True)
print('Seed-VC klar.' if ok else 'Seed-VC hoppas över (valfritt) – motorn startar ändå, bara röstbyte saknas. Logg: ' + L)

## 6. Starta server + publik tunnel

In [ ]:
import os, secrets, sys, threading, subprocess, re, time, json, urllib.request
from IPython.display import display, HTML
SRV = '/tmp/kino/kino_colab_server.py'
if not os.path.exists(SRV):  # självläkande: hämta servern om cell 3 inte körts
    try:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/pistpejl/kino/main/colab/kino_colab_server.py', SRV)
        print('Server hämtad:', os.path.getsize(SRV), 'byte')
    except Exception as e:
        raise SystemExit(f'Kunde inte hämta servern ({e}). Är Internet på? Settings → Internet → On, och kör om cellen.')
os.environ['KINO_TOKEN'] = secrets.token_urlsafe(24)   # ny slumpad token varje körning
os.environ['KINO_PLATFORM'] = 'kaggle'; os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
try:  # valfritt: HF_TOKEN i Add-ons → Secrets, bara för FLUX 3 Image Klein (gated). Skrivs aldrig ut.
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN') or ''
except Exception: pass
for p in list(globals().get('_kino_procs', [])):  # kör man cellen igen: stäng förra servern/tunneln
    try: p.terminate(); p.wait(10)
    except Exception: pass
# Egen process (python -m uvicorn) så att en förladdad uvicorn i kärnan aldrig krockar med den installerade
srv = subprocess.Popen([sys.executable, '-u', '-m', 'uvicorn', 'kino_colab_server:app', '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
                       cwd='/tmp/kino', env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
_srv_log = []
def _pump():
    for line in srv.stdout:
        _srv_log.append(line); del _srv_log[:-200]; print('[server]', line, end='')
threading.Thread(target=_pump, daemon=True).start()
_kino_procs = [srv]
def health():
    with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5) as r: return json.load(r)
for _ in range(150):
    if srv.poll() is not None:
        time.sleep(1); raise RuntimeError('Servern kraschade vid start (kod %s):\n' % srv.returncode + ''.join(_srv_log[-40:]))
    try: h = health(); break
    except Exception: time.sleep(2)
else: raise RuntimeError('Servern svarade inte på /health inom 5 min:\n' + ''.join(_srv_log[-40:]))
time.sleep(2)
if srv.poll() is not None:  # t.ex. port 8000 upptagen av en gammal server: starta om sessionen
    raise RuntimeError('Servern stängdes direkt (kod %s):\n' % srv.returncode + ''.join(_srv_log[-40:]))
print('Server uppe | GPU:', h.get('gpu'), '| funktioner:', ', '.join(h.get('features', [])))
TOKEN = os.environ['KINO_TOKEN']
cf = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
_kino_procs.append(cf)
url, t0 = None, time.time()
for line in cf.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: url = m.group(0); break
    if time.time() - t0 > 120: break
if not url: raise RuntimeError('cloudflared gav ingen adress. Kör cellen igen.')
threading.Thread(target=lambda: [None for _ in cf.stdout], daemon=True).start()
display(HTML(f'''<div style="font-family:sans-serif;padding:16px;border:3px solid #2a2;border-radius:12px">
<div style="font-size:16px">URL (klistra in i KINO):</div><div style="font-size:26px;font-weight:bold;user-select:all">{url}</div>
<div style="font-size:16px;margin-top:12px">Token:</div><div style="font-size:22px;font-weight:bold;user-select:all">{TOKEN}</div></div>'''))
print('URL:  ', url); print('TOKEN:', TOKEN)


## 7. Håll igång
Kör kvar så länge du använder motorn. Stop/stäng sessionen = motorn stängs.

In [ ]:
import time, json, urllib.request
def _get(path):
    rq = urllib.request.Request('http://127.0.0.1:8000' + path, headers={'Authorization': 'Bearer ' + TOKEN})
    with urllib.request.urlopen(rq, timeout=10) as r: return json.load(r)
if 'srv' not in globals(): raise SystemExit('Servern startades inte: kör cell 6 (Starta server) först och läs felet där.')
while True:
    if srv.poll() is not None: raise RuntimeError('Servern har stängts (kod %s). Kör cell 6 igen.\n' % srv.returncode + ''.join(_srv_log[-40:]))
    if cf.poll() is not None: print('VARNING: tunneln (cloudflared) har stängts – kör cell 6 igen för ny adress.')
    try:
        h, j = _get('/health'), _get('/jobs')
        print(time.strftime('%H:%M'), '| kö:', h['queue'], '| laddad:', h['loaded'], '| aktiva jobb:', [x['kind'] + ':' + x['status'] for x in j['jobs']])
    except Exception as e: print(time.strftime('%H:%M'), '| status ej tillgänglig:', e)
    time.sleep(60)
